# GraphRAG

## Import packages

In [5]:
import sys
sys.path.append('..')
sys.path.append('../neurorag')
sys.path.append('../neurorag/chains')

import os
import nltk
import string
import numpy as np
import pandas as pd
from unidecode import unidecode
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
from pathlib import Path
from rouge_score import rouge_scorer
import json
from dotenv import load_dotenv
from getpass import getpass

from langchain_community.embeddings import OllamaEmbeddings
from langchain.embeddings.cache import CacheBackedEmbeddings
from langchain.storage import LocalFileStore

from neurorag.neurorag import NeuroRAG

## Disable warnings

In [6]:
import warnings
warnings.filterwarnings('ignore')

## Setup environment variables

You have to define the following environment variables in the `.env` file, terminal environment, or input field within this Jupyter notebook:
1. MISTRAL_API_KEY
2. OPENAI_API_KEY
3. OPENAI_PROXY
4. TAVILY_API_KEY
5. ENTREZ_EMAIL

## Import packages

In [7]:
env_variables = [
  'MISTRAL_API_KEY',
  'OPENAI_API_KEY',
  'OPENAI_PROXY',
  'TAVILY_API_KEY',
  'ENTREZ_EMAIL',
]

load_dotenv()

for key in env_variables:
  value = os.getenv(key)

  if value is None:
    value = getpass(key)

  os.environ[key] = value

## Setup metrics

### Download NLTK dictionaries

These dictionaries are needed for further text preprocessing.

In [8]:
dict_ids = [
  'punkt_tab',
  'punkt',
  'stopwords',
  'wordnet',
]

for dict_id in dict_ids:
  nltk.download(dict_id, quiet=True)

### Text preprocessing

Define a function for text preprocessing, which is an important step before calculating any metrics. This preprocessing function will help in cleaning the text data, making it ready for further analysis. The preprocessing involves several steps:
1. Lowercasing
2. Stopwords removal
3. Lemmatization
4. Remove accents from characters

In [9]:
lemmatizer = nltk.stem.WordNetLemmatizer()

def preprocess(corpus: str) -> str:
  corpus = corpus.lower()
  stopset = nltk.corpus.stopwords.words('english') + nltk.corpus.stopwords.words('russian') + list(string.punctuation)
  tokens = nltk.word_tokenize(corpus)
  tokens = [t for t in tokens if t not in stopset]
  tokens = [lemmatizer.lemmatize(t) for t in tokens]
  corpus = ' '.join(tokens)
  corpus = unidecode(corpus)
  return corpus

### Embedding Initialization

Here we are initializing the Llama 3 embeddings model. The `OllamaEmbeddings` class is a component of the Ollama library, a set of pre-trained language models. This model is capable of embedding corpora of any length into a 4096-dimensional vector.

The use of `OllamaEmbeddings` requires the installation of a local Ollama server, which can be found at https://ollama.com.

In [10]:
embeddings = OllamaEmbeddings(model='llama3.1')
store = LocalFileStore("./.embeddings_cache")

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
  embeddings,
  store,
  namespace=embeddings.model,
)

### Average embeddings cosine similarity metric

This function calculates the average cosine similarity between expected answers and LLM predicted answers using their respective embeddings. Cosine similarity is a measure of similarity between two non-zero vectors of an inner product space that measures the cosine of the angle between them:

$$
K(a, b) = \frac{\sum \limits_{i=1}^n a_i b_i}{\sqrt{\sum \limits_{i=1}^n a_i^2} \cdot \sqrt{\sum \limits_{i=1}^n b_i^2}}
$$

In [11]:
def embeddings_cosine_sim_metric(expected_answers: list[str], predicted_answers: list[str]) -> float:
  results = []

  for expected_answer, predicted_answer in zip(expected_answers, predicted_answers):
    expected_answer = preprocess(expected_answer)
    predicted_answer = preprocess(predicted_answer)

    expected_embedding = np.array(cached_embeddings.embed_query(expected_answer))
    predicted_embedding = np.array(cached_embeddings.embed_query(predicted_answer))

    sim = cosine_similarity(
      expected_embedding.reshape(1, -1),
      predicted_embedding.reshape(1, -1),
    )[0][0]

    results.append(sim)

  return np.mean(results)

In [12]:
smoothie_f = nltk.translate.bleu_score.SmoothingFunction().method4

def bleu_metric(expected_answers, predicted_answers):
  scores = []

  for expected_answer, predicted_answer in zip(expected_answers, predicted_answers):
    expected_answer = preprocess(expected_answer)
    predicted_answer = preprocess(predicted_answer)

    predicted_tokens = nltk.word_tokenize(predicted_answer)
    expected_tokens = [nltk.word_tokenize(expected_answer)]

    score = nltk.translate.bleu_score.sentence_bleu(
      expected_tokens,
      predicted_tokens,
      smoothing_function=smoothie_f,
    )

    scores.append(score)

  return np.mean(scores)

In [13]:
rogue_1_scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)

def rogue_1_metric(expected_answers, predicted_answers):
  scores = []

  for expected_answer, predicted_answer in zip(expected_answers, predicted_answers):
    expected_answer = preprocess(expected_answer)
    predicted_answer = preprocess(predicted_answer)

    result = rogue_1_scorer.score(expected_answer, predicted_answer)

    scores.append(result['rouge1'])

  return np.mean(scores)

In [14]:
rogue_l_scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def rogue_l_metric(expected_answers, predicted_answers):
  scores = []

  for expected_answer, predicted_answer in zip(expected_answers, predicted_answers):
    expected_answer = preprocess(expected_answer)
    predicted_answer = preprocess(predicted_answer)

    result = rogue_l_scorer.score(expected_answer, predicted_answer)

    scores.append(result['rougeL'])

  return np.mean(scores)

## Build model

In [15]:
app = NeuroRAG(debug=False)
app.compile()

## Evaluate RAG

### Load QA dataset

In [16]:
qa_df = pd.read_csv('../datasets/brainscape.csv')
qa_df

,question,answer
0,What are the afferent cranial nerve nuclei?,Trigeminal sensory nucleus- fibres carry gener...
1,What is the order of the cranial nerves ?,1-olfactory\n2-optic\n3-oculomotor\n4-trochlea...
2,What are the efferent cranial nerve nuclei?,Edinger-westphal nucleus\nOculomotor nucleus\n...
3,Which nuclei share the embryo logical origin -...,Oculomotor nucleus Trochlear nucleus Abducens ...
4,Which nuclei share the embryo logical origin- ...,Trigeminal motor nucleus Facial motor nucleus ...
...,...,...
1047,What is the purpose of gephyrin in the glycine...,Involved in anchoring the receptor to a specif...
1048,What is the glycine receptor involved in ?,Reflex response\nCauses reciprocal inhibition ...
1049,What happens in hyperperplexia ?,It’s an exaggerated reflex Often caused by a m...
1050,What is hyperperplexia treated with ?,Benzodiazepine


### Load cached RAGs responses

In [17]:
cache_path = Path('cache.json')

if not os.path.exists(cache_path):
  data = {}
  with open(cache_path, 'w') as file:
    json.dump(data, file)

with open(cache_path, 'r') as f:
  cache = json.load(f)

len(cache.keys())

4

In [18]:
questions = [
    'Describe Minian software for miniscope data analysis',
]

for index, question in enumerate(questions):
    response = app.invoke(question)
    generation = response['generation']
    documents = response['documents']
    sources = [
      document.metadata['source']
      for document in documents
      if 'source' in document.metadata
    ]

    print(f'{index + 1}. {question}')
    print(generation)
    print(sources)
    print('')

here 0
web_results [Document(metadata={'source': 'https://edspace.american.edu/openbehavior/project/minian/'}, page_content='Minian is a user friendly miniscope analysis pipeline that requires low memory and computational demand such that it can be run without specialized hardware. Minion offers interactive visualization that allows users to see how parameters in each step of the pipeline affect the output. This functionality allows users with little computational knowledge to produce high quality calcium imaging results. They also provide detailed documentation and have validated their pipeline across multiple brain [...] regions and neuron types. Minian is thus a great entry point for labs looking to implement miniscopes that are unsure how to choose the best parameters for their specific use case. [...] With increasing interest in miniscopes as a method for in vivo calcium imaging, there is need for tools that increase accessibility to the computationally complex process of analysin

In [19]:
response

{'query': 'Describe Minian software for miniscope data analysis',
 'specialized_sources': ['vectorstore'],
 'step_back_query': 'What is Minian software used for?',
 'rewritten_query': 'Describe Minian software for miniscope data analysis',
 'subqueries': ['What are the key features of Minian software for miniscope data analysis?',
  'How does Minian software handle image processing and enhancement for miniscope data?',
  'What are the benefits of using Minian software for miniscope data analysis in neuroscience research?',
  'Can Minian software be integrated with other tools and platforms for comprehensive miniscope data analysis?'],
 'generated_documents': ["Here is a potential scientific paper passage answering the question:\n\n**Title:** Minian Software for Miniscope Data Analysis: A Comprehensive Framework for High-Resolution Imaging and Behavioral Studies\n\n**Abstract:**\nThe Miniscope, a miniature microscope, has revolutionized the field of neuroscience by enabling high-resolut

In [20]:
# questions = list(qa_df['question'].tolist())
# expected_answers = list(qa_df['answer'].tolist())
# predicted_answers = []

# for index, question in tqdm(enumerate(questions)):
#   if not question in cache:
#     cache[question] = app.invoke({'question': question})['generation']

#   predicted_answers.append(cache[question])

#   with open(cache_path, 'w') as f:
#     json.dump(cache, f)

# cos_score = embeddings_cosine_sim_metric(expected_answers, predicted_answers)
# bleu_score = bleu_metric(expected_answers, predicted_answers)
# rogue_1_score = rogue_1_metric(expected_answers, predicted_answers)
# rogue_l_score = rogue_l_metric(expected_answers, predicted_answers)

# cos_score, bleu_score, rogue_1_score, rogue_l_score